# CapAI — Water Polo Ball Detector Training
**Model:** YOLO26l (large)  
**Dataset:** Water polo ball from Roboflow Universe  
**Runtime:** GPU → T4 (Runtime > Change runtime type > T4 GPU)

Run cells top to bottom. At the end you'll download `best.pt` — drop it into `backend/ball_detector.pt` in your CapAI repo.

> YOLO26 is NMS-free and trains/exports with the same Ultralytics API as v8/v11. It's faster on CPU (your HF backend has no GPU), so inference per `/analyze` request is quicker.

In [ ]:
# Cell 1 — Check GPU
!nvidia-smi

In [ ]:
# Cell 2 — Install dependencies
# YOLO26 needs a recent ultralytics; -U pulls the latest that ships the weights.
!pip install -U ultralytics roboflow --quiet

In [ ]:
# Cell 3 — Download dataset from Roboflow
# Steps to get your values:
#   1. Go to https://universe.roboflow.com and search "water polo ball"
#   2. Pick a dataset (e.g. the one by user 'sport-analytics' or similar)
#   3. Click Download > pick YOLO11 (or YOLOv8) format > Show download code
#   4. Copy your api_key, workspace, project, and version from that snippet
#
# NOTE: YOLO26 uses the SAME label format as YOLOv8/YOLO11, so any of those
# export options works identically. "yolov8" below is just the format name.

from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY_HERE")  # <-- paste your key
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)  # change version number if needed
dataset = version.download("yolov8")  # same format YOLO26 trains on

print("Dataset location:", dataset.location)

In [ ]:
# Cell 4 — Peek at the dataset
import os, yaml

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml) as f:
    info = yaml.safe_load(f)

print(yaml.dump(info))

# Count images
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    if os.path.exists(img_dir):
        print(f"{split}: {len(os.listdir(img_dir))} images")

In [ ]:
# Cell 5 — Train YOLO26l
# T4 has 16GB VRAM — YOLO26l fits at batch=16 comfortably.
# 150 epochs is enough for a focused ball detector; early stopping
# exits if val loss stops improving for 30 consecutive epochs.

from ultralytics import YOLO

model = YOLO("yolo26l.pt")  # downloads pretrained weights automatically

results = model.train(
    data=data_yaml,
    epochs=150,
    patience=30,          # early stop after 30 epochs with no improvement
    batch=16,
    imgsz=640,
    device=0,             # GPU
    project="capai_ball",
    name="yolo26l_waterpolo",
    exist_ok=True,
    # Augmentations (all on by default in ultralytics, tuned here for pool)
    hsv_h=0.015,          # slight hue shift (ball is distinctly yellow)
    hsv_s=0.5,            # saturation variation (wet ball / glare)
    hsv_v=0.4,            # brightness variation (indoor/outdoor pools)
    flipud=0.0,           # no vertical flip (balls don't appear upside down)
    fliplr=0.5,           # horizontal flip OK
    mosaic=1.0,           # mosaic augmentation helps small object detection
    degrees=5.0,          # slight rotation
)

print("\nBest model saved to:", results.save_dir)

In [ ]:
# Cell 6 — Evaluate on validation set
best_weights = f"{results.save_dir}/weights/best.pt"
best_model = YOLO(best_weights)
metrics = best_model.val(data=data_yaml, device=0)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
# Cell 7 — Quick visual check on a few val images
import glob
from IPython.display import Image, display

val_images = glob.glob(os.path.join(dataset.location, "valid", "images", "*.jpg"))[:4]
best_model.predict(val_images, save=True, project="capai_ball", name="preview", exist_ok=True)

for img in glob.glob("capai_ball/preview/*.jpg")[:4]:
    display(Image(img, width=480))

In [ ]:
# Cell 8 — Download best.pt to your computer
# Then drop it into backend/ball_detector.pt in your CapAI repo
from google.colab import files
files.download(best_weights)
print("Downloaded:", best_weights)